# 03d Fine-tuning


In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)


In [ ]:
import pickle

config = load_experiment_config()
fractions = config['finetuning']['fractions']

model_dir = OUTPUT_DIR / 'models'
model_path = model_dir / 'berlin_ml_champion.pkl'
metadata_path = model_path.with_suffix('.pkl.metadata.json')

leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(DATA_DIR / 'phase_3_experiments')
feature_cols = data_loading.get_feature_columns(leipzig_finetune)

with (model_dir / 'berlin_scaler.pkl').open('rb') as handle:
    scaler = pickle.load(handle)

metadata = json.loads(metadata_path.read_text())
label_to_idx = {k: int(v) for k, v in metadata['label_to_idx'].items()}

results = []
for frac in fractions:
    sample = leipzig_finetune.sample(frac=frac, random_state=config['global']['random_seed'])
    x_train = scaler.transform(sample[feature_cols].to_numpy(dtype=float))
    x_test = scaler.transform(leipzig_test[feature_cols].to_numpy(dtype=float))

    y_train = sample['genus_latin'].map(label_to_idx).to_numpy()
    y_test = leipzig_test['genus_latin'].map(label_to_idx).to_numpy()

    model = models.create_model('random_forest')
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    metrics = evaluation.compute_metrics(y_test, preds)
    results.append({'fraction': frac, 'model_type': 'ml', 'metrics': metrics, 'f1_score': metrics['f1_score']})

curve_path = OUTPUT_DIR / 'metadata/finetuning_curve.json'
curve_path.write_text(json.dumps({'results': results}, indent=2))
validate_finetuning_curve(curve_path)

fig_dir = OUTPUT_DIR / 'figures/finetuning'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_finetuning_curve(
    [{'fraction': r['fraction'], 'f1_score': r['metrics']['f1_score']} for r in results],
    baselines={},
    output_path=fig_dir / 'finetuning_curve.png',
)
